# 04 — Competitive Analysis

Loads pre-computed competitive features and explores the competitive landscape:
how many competitors exist per product, relative price positioning, and
competitor price change activity.

**Prerequisites**:
- `../data/features/competitive_features.csv`  (from `scripts/generate_queries.py`)

**Outputs**
- `../data/figures/products_most_competitors.png`
- `../data/figures/multi_merchant_price_history.png`
- `../data/figures/price_vs_comp_median_dist.png`
- `../data/figures/competitors_changed_7d.png`

In [ ]:
import sys
import os
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import numpy as np

warnings.filterwarnings('ignore')

sys.path.insert(0, '..')

from src.features.competitive_features import load_competitive_features

os.makedirs('../data/figures', exist_ok=True)

print('Imports OK')

## 1. Load Competitive Features

In [ ]:
features_path = '../data/features/competitive_features.csv'

try:
    cf = load_competitive_features(features_path)
    print(f'Loaded competitive features: {cf.shape[0]:,} rows × {cf.shape[1]} columns')
    print(f'\nColumns: {list(cf.columns)}')
except FileNotFoundError:
    raise FileNotFoundError(
        f'Feature file not found at {features_path}.\n'
        'Run scripts/generate_queries.py (or scripts/run_bocpd.py) to produce this file first.'
    )

# Quick sanity check
print(f'\nDate range: {pd.to_datetime(cf["observed_at"]).min()} → {pd.to_datetime(cf["observed_at"]).max()}')
print(f'Unique products  : {cf["product_id"].nunique():,}')
print(f'Unique merchants : {cf["merchant"].nunique():,}')
print(f'Unique PM pairs  : {cf.groupby(["product_id","merchant"]).ngroups:,}')

## 2. Products With the Most Competitors

In [ ]:
# Maximum competitor_count seen per product (peak competitive pressure)
max_competitors = (
    cf
    .groupby('product_id')['competitor_count']
    .max()
    .rename('max_competitor_count')
    .reset_index()
    .nlargest(15, 'max_competitor_count')
    .sort_values('max_competitor_count', ascending=True)
)

print('Top-15 products by maximum competitor count:')
display(max_competitors.reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(max_competitors)))
bars = ax.barh(
    max_competitors['product_id'].apply(lambda x: str(x)[:40]),
    max_competitors['max_competitor_count'],
    color=colors
)
ax.set_xlabel('Max Competitor Count', fontsize=12)
ax.set_title('Products With the Most Competitors (Peak Count)', fontsize=14, fontweight='bold')
for bar, val in zip(bars, max_competitors['max_competitor_count']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
            str(int(val)), va='center', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/figures/products_most_competitors.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: ../data/figures/products_most_competitors.png')

## 3. Multi-Merchant Price History for a Sample Product

In [ ]:
# Pick the product with the most distinct merchants for a rich comparison chart
merchant_counts = (
    cf
    .groupby('product_id')['merchant']
    .nunique()
    .rename('n_merchants')
)
sample_pid = merchant_counts.idxmax()
n_merchants_sample = int(merchant_counts[sample_pid])
print(f'Selected product: {sample_pid}')
print(f'Number of merchants: {n_merchants_sample}')

product_data = cf[cf['product_id'] == sample_pid].copy()
product_data['observed_at'] = pd.to_datetime(product_data['observed_at'])

fig, ax = plt.subplots(figsize=(13, 6))

merchants_in_product = product_data['merchant'].unique()
colors_map = plt.cm.tab10.colors if len(merchants_in_product) <= 10 else plt.cm.tab20.colors

for idx, m in enumerate(merchants_in_product):
    m_data = (
        product_data[product_data['merchant'] == m]
        .sort_values('observed_at')
    )
    color = colors_map[idx % len(colors_map)]
    ax.plot(
        m_data['observed_at'],
        m_data['price'],
        marker='o', markersize=3, linewidth=1.1,
        color=color,
        label=str(m)[:30]
    )

short_pid = str(sample_pid)[:50] + ('...' if len(str(sample_pid)) > 50 else '')
ax.set_title(f'Price Histories — All Merchants\n{short_pid}', fontsize=12, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Price (USD)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.2f'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.legend(fontsize=8, loc='upper right', framealpha=0.7,
          ncol=max(1, n_merchants_sample // 5))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../data/figures/multi_merchant_price_history.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: ../data/figures/multi_merchant_price_history.png')

## 4. Distribution of `price_vs_comp_median`

In [ ]:
pvc_col = 'price_vs_comp_median'

if pvc_col not in cf.columns:
    print(f'Column {pvc_col} not found. Available columns: {list(cf.columns)}')
else:
    pvc = cf[pvc_col].dropna()
    # Clip to ±100% for readability
    pvc_clip = pvc.clip(-100, 100)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(pvc_clip, bins=80, color='mediumpurple', edgecolor='white', linewidth=0.3)
    ax.axvline(0, color='black', linewidth=1.2, linestyle='--', label='0 (at median)')
    ax.axvline(pvc.median(), color='red', linewidth=1.1, linestyle='-.',
               label=f'Overall median {pvc.median():.3f}')
    ax.set_xlabel('Price vs Competitor Median (relative, clipped ±100%)', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title('Distribution of price_vs_comp_median', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)

    # Annotate % below / above 0
    pct_below = (pvc < 0).mean() * 100
    pct_above = (pvc > 0).mean() * 100
    ax.text(0.02, 0.95, f'{pct_below:.1f}% below median (cheaper)',
            transform=ax.transAxes, fontsize=9, color='green', va='top')
    ax.text(0.98, 0.95, f'{pct_above:.1f}% above median (pricier)',
            transform=ax.transAxes, fontsize=9, color='red', va='top', ha='right')

    plt.tight_layout()
    plt.savefig('../data/figures/price_vs_comp_median_dist.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Saved: ../data/figures/price_vs_comp_median_dist.png')
    print(f'\n{pvc_col} stats (unclipped):')
    print(pvc.describe().to_string())

## 5. Events Where `competitors_changed_7d > 0`

In [ ]:
cc_col = 'competitors_changed_7d'

if cc_col not in cf.columns:
    print(f'Column {cc_col} not found. Available columns: {list(cf.columns)}')
else:
    total_obs = len(cf)
    nonzero_mask = cf[cc_col].fillna(0) > 0
    n_nonzero = nonzero_mask.sum()
    pct_nonzero = n_nonzero / total_obs * 100

    print(f'Total observations             : {total_obs:,}')
    print(f'Observations with {cc_col} > 0 : {n_nonzero:,} ({pct_nonzero:.1f}%)')

    # Distribution of the count values
    value_counts = cf[cc_col].fillna(0).astype(int).value_counts().sort_index()
    print(f'\nValue distribution (top 10 values):')
    display(value_counts.head(10).rename('count').reset_index().rename(columns={'index': cc_col}))

    # Bar chart: 0 vs 1 vs 2 vs 3+
    def bucket_cc(n):
        if n == 0:
            return '0'
        elif n == 1:
            return '1'
        elif n == 2:
            return '2'
        else:
            return '3+'

    bucket_order = ['0', '1', '2', '3+']
    cc_buckets = (
        cf[cc_col].fillna(0).astype(int)
        .apply(bucket_cc)
        .value_counts()
        .reindex(bucket_order, fill_value=0)
    )

    fig, ax = plt.subplots(figsize=(7, 5))
    colors_cc = ['#95a5a6', '#3498db', '#e67e22', '#e74c3c']
    bars = ax.bar(cc_buckets.index, cc_buckets.values, color=colors_cc,
                  edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, cc_buckets.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
                f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_xlabel('competitors_changed_7d (bucket)', fontsize=12)
    ax.set_ylabel('Observation Count', fontsize=12)
    ax.set_title('Observations by Competitor Activity in Last 7 Days', fontsize=13, fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../data/figures/competitors_changed_7d.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Saved: ../data/figures/competitors_changed_7d.png')

## 6. Summary: is_cheapest / is_most_expensive

In [ ]:
for col in ['is_cheapest', 'is_most_expensive']:
    if col in cf.columns:
        valid = cf[col].dropna()
        pct_true = valid.astype(float).mean() * 100
        print(f'{col}: {pct_true:.1f}% of competitive observations are True')
    else:
        print(f'{col}: column not found')